# 1.HumanInTheLoopMiddleware中间件

## 1.1举例的过程1：工具调用的中断

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

summary_model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
    profile = {"max_input_tokens":64000}   # 务必显式赋值
)

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from rich import print as rprint


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent = create_agent(
    model = "deepseek-flash",
    tools = [get_news, get_weather, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather":True,
                "get_news":True,
                "read_email_tool":False,
                "send_email_tool":{
                    "allowed_decisions":["approve", "reject"],
                    "description":"发送邮件中断了！"
                }
            },
            description_prefix="中断了！"
        )
    ]
)

config = {"configurable":{"thread_id":"1"}}

# 第一次调用：会暂停在发送邮件前
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请帮我查询今天北京的天气"
                                 "查询今日新闻"
                                 "查看ID为 'sk2131421' 的邮件内容，"
                                 "内容是：'你好啊'"
                                 "向15641685664@qq.com发送邮件，标题是'哈哈'，内"
                                 "同时做这四件事")
        ]
    },
    config=config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)
print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()

# 关键：看中断信息
interrupts = response.get("__interrupt__", [])
print("========= interrupts =========")
rprint(interrupts)

==== 第一次 invoke 返回 ====
========= 原始响应 =========


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，内容是：'你好啊'向15641685664@qq.com发送邮件，标题是'哈哈'，内同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='a24cd247-6b50-4704-a07d-e9f193d93be4'
        ),
        AIMessage(
            content='我来帮您同时完成这四件事。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'The user wants me to do four things:\n1. Query today\'s Beijing weather\n2. 
Query today\'s news\n3. Read email with ID \'sk2131421\', content is \'你好啊\'\n4. Send email to 
15641685664@qq.com, title \'哈哈\'\n\nLet me do them all in parallel since they\'re independent.\n\nActually, wait 
- the email reading content is stated as \'你好啊\'? The user says "查看ID为 \'sk2131421\' 
的邮件内容，内容是：\'你好啊\'" - this seems to say the email content is \'你好啊\'. Hmm, but that\'s odd. Let me 
just read the email.\n\nLet me make independent calls in the same block.'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 314,
                    'prompt_tokens': 533,
                    'total_tokens': 847,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 150,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 149
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '665e807f-7ea4-4e86-9463-d9eba3795d78',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a13a64-4fb6-7442-8642-284e1ad4bde5-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_Qf3nmYchWjeBwSLvver81351',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_j1P5h4j7lbIfHpGKz9qP2123', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_02_BKHMF93VzPfOZ1Ika1R19775',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈', 'body': '你好啊'},
                    'id': 'call_03_6g2XSULLnAb1OQxOfNYQ0741',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 533,
                'output_tokens': 314,
                'total_tokens': 847,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {'reasoning': 150}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京'},
                        'description': "中断了！\n\nTool: get_weather\nArgs: {'city': '北京'}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断了！\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断了！'
                  

========= 美化输出 =========
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，内容是：'你好啊'向15641685664@qq.com发送邮件，标题是'哈哈'，内同时做这四件事
================================== Ai Message ==================================

我来帮您同时完成这四件事。
Tool Calls:
  get_weather (call_00_Qf3nmYchWjeBwSLvver81351)
 Call ID: call_00_Qf3nmYchWjeBwSLvver81351
  Args:
    city: 北京
  get_news (call_01_j1P5h4j7lbIfHpGKz9qP2123)
 Call ID: call_01_j1P5h4j7lbIfHpGKz9qP2123
  Args:
  read_email_tool (call_02_BKHMF93VzPfOZ1Ika1R19775)
 Call ID: call_02_BKHMF93VzPfOZ1Ika1R19775
  Args:
    email_id: sk2131421
  send_email_tool (call_03_6g2XSULLnAb1OQxOfNYQ0741)
 Call ID: call_03_6g2XSULLnAb1OQxOfNYQ0741
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈
    body: 你好啊
========= interrupts =========


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'description': "中断了！\n\nTool: get_weather\nArgs: {'city': '北京'}"
                },
                {'name': 'get_news', 'args': {}, 'description': '中断了！\n\nTool: get_news\nArgs: {}'},
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈', 'body': '你好啊'},
                    'description': '发送邮件中断了！'
                }
            ],
            'review_configs': [
                {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='f033f716152468f8b3d39981885f71c8'
    )
]

## 举例的过程2：指明工具调用请求策略

In [9]:
from langgraph.types import Command

weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args":{"city": "上海市", "is_forcast": True}
    }
}

news_decision = {
    "type":"approve"
}

send_email_decision = {
    "type":"approve"
}


decisions = {
    "decisions": []
}

interrupts = response.get("__interrupt__", [])
action_requests = interrupts[0].value["action_requests"]

for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts:
    resumed_res = agent.invoke(
        Command(resume=decisions),
        config = config
    )

    for msg in resumed_res["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，内容是：'你好啊'向15641685664@qq.com发送邮件，标题是'哈哈'，内同时做这四件事
================================== Ai Message ==================================

我来帮您同时完成这四件事。
Tool Calls:
  get_weather (call_00_Qf3nmYchWjeBwSLvver81351)
 Call ID: call_00_Qf3nmYchWjeBwSLvver81351
  Args:
    city: 上海市
    is_forcast: True
  get_news (call_01_j1P5h4j7lbIfHpGKz9qP2123)
 Call ID: call_01_j1P5h4j7lbIfHpGKz9qP2123
  Args:
  read_email_tool (call_02_BKHMF93VzPfOZ1Ika1R19775)
 Call ID: call_02_BKHMF93VzPfOZ1Ika1R19775
  Args:
    email_id: sk2131421
  send_email_tool (call_03_6g2XSULLnAb1OQxOfNYQ0741)
 Call ID: call_03_6g2XSULLnAb1OQxOfNYQ0741
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

上海市今天天气不错
明天下雨
================================= Tool Message ============